# E-SEEDEXT2 — extending the primary bound from six to twelve adapters per arm

**Run all. ~11 GPU-hours. Fully resumable — your existing six adapters per arm are reused,
not retrained.**

## Pre-declaration — read this before running anything

> We extend from six to twelve independently seeded adapters per arm using seeds 6–11,
> continuing the existing sequence. The extension is declared before any new adapter is
> trained and before any new measurement is examined. The inferential procedure is
> unchanged: the intersection–union sign-flip test at $\alpha = 0.01$ is primary, with the
> simultaneous upper confidence limit reported alongside. No adapter is excluded on the
> basis of its measured contrast. Whatever the resulting limit, it is reported.

**Do not run stage S4 until S1 through S3 have finished for both arms.** The value of this
extension comes from not having looked.

## Why twelve and not eight or twenty

| k | BCa coverage | $\lambda_U$ | one adapter can move $U$ by |
|---|---|---|---|
| 6 | 98.5 % marginal | 0.249 % | **72 %** |
| 8 | 98.8 % marginal | 0.198 % | 53 % |
| 10 | 99.3 % valid | 0.172 % | 45 % |
| **12** | **99.5 % valid** | **0.155 %** | **36 %** |
| 20 | 99.3 % | 0.123 % | 23 % |

Four things change qualitatively at twelve:

**BCa stops being marginal.** It is the headline construction, and at $k=6$ it covers at
98.5 % under heavy-tailed clusters — it survives only because the reported maximum dominates
the plug-in limit. At twelve it is valid on its own.

**Single-adapter leverage halves.** At $k=6$ one run can move the bound by 72 %, which is
why `A_raw_s5` is a standing concern and why a leave-one-out sensitivity has to be
disclosed.

**The cluster distribution becomes testable.** Six points cannot distinguish a normal from a
$t_3$, so the coverage simulation has to assume shapes. Twelve can estimate one. Stage S4
runs that test.

**The registered test becomes capable of rejecting.** The exact sign-flip floor is $2^{-k}$:
at $k=6$ that is $1/64 = 0.0156 > 0.01$, so the registered test could never reject whatever
the data said. At $k=12$ it is $1/4096$. "The registered test could reject at 0.01 and did
not, across twelve independently seeded adapters per arm" is a sentence the study cannot
currently write.

## The honest risk

The projected $\lambda_U = 0.155\,\%$ assumes the new adapters have the same spread as the
six you have. If `s5` was a fluke the limit lands near 0.124 %; if outliers occur about one
in six it stays near 0.155 %; **if the new adapters are more variable the limit loosens.**
All three are informative. A limit that widened on more data, reported plainly, is worth
more than a tight one from six clusters that no reviewer believes.

In [ ]:
# %% 1 · mount, Hugging Face login, imports, configuration
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random, itertools
import csv as _csv
from dataclasses import dataclass, asdict

HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive"
except Exception:
    DRIVE = os.environ.get("EINV_DRIVE", ".")

@dataclass
class CFG:
    STAGES: tuple = ("S0",)                        # S0 -> S1 -> S2 -> S3 -> S4
    BASE:  str = "inv_channel/E_INV_P0_v3"         # seeds 0-2, fingerprints, manifest
    PREV:  str = "inv_channel/E_SEEDEXT"           # seeds 3-5
    OUT:   str = "inv_channel/E_SEEDEXT2"          # seeds 6-11, written here
    ARMS:  tuple = ("A", "B")
    NEW_SEEDS: tuple = (6, 7, 8, 9, 10, 11)
    # these MUST match the base study; stage S0 verifies them by regenerating a stored image
    CAPTION: str = "a photograph, sks style"
    GEN_SEED_BASE: int = 770000
    SD: str = "stabilityai/stable-diffusion-3.5-medium"
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    G_PER: int = 500; GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 2
    CROP: int = 1024
    ALPHA: float = 0.01
    GRAD_CKPT: bool = False   # True halves S1 activation memory, costs ~30% speed
C = CFG(); MEAS = C.CROP
BASE = os.path.join(DRIVE, C.BASE); PREV = os.path.join(DRIVE, C.PREV)
ROOT = os.path.join(DRIVE, C.OUT)
for d in ("", "adapters", "gens", "csv", "meta"): os.makedirs(os.path.join(ROOT, d), exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
def HALT(m): raise SystemExit(f"\n{'='*72}\n[HALT] {m}\n{'='*72}")

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],
               check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _p1; _p1.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as _p2; _p2.is_torchao_available = lambda: False
except Exception: pass
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DT  = torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN); print("[env] HF login ok")

if not os.path.isdir(BASE): HALT(f"base study root not found: {BASE}")
FP = {}
for r in C.ARMS:
    p = os.path.join(BASE,"fingerprints",f"K_{r}_E2.npy")
    if not os.path.exists(p): HALT(f"fingerprint missing: {p}")
    FP[r] = np.load(p)
MAN = {}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((str(r["role"] if "role" in r else r.get("dev")), r["split"]), []).append(r["path"])
def tdir(a): return os.path.join(BASE, "train_png", a)
def tag(a,s): return f"{a}_raw_s{s}_r16"
def adir(t): return os.path.join(ROOT,"adapters",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
NEW = [(tag(a,s), a, s) for s in C.NEW_SEEDS for a in C.ARMS]
def append_row(p,h,r):
    new = not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o, open(t,"w"), **kw); os.replace(t,p)
print(f"[cfg] base {BASE}\n[cfg] {len(NEW)} new adapters: seeds {C.NEW_SEEDS} x arms {C.ARMS}")
print(f"[cfg] target k = {6 + len(C.NEW_SEEDS)} per arm  "
      f"(sign-flip floor 1/{2**(6+len(C.NEW_SEEDS))})")
if DEV == "cuda":
    _tot = torch.cuda.get_device_properties(0).total_memory/2**30
    print(f"[env] {torch.cuda.get_device_name(0)}  {_tot:.0f} GB  dtype {DT}")
    if _tot < 45 and not C.GRAD_CKPT:
        print("[env] *** On a 40 GB card S1 peaks near 28 GB at batch 2 without\n      gradient checkpointing. That fits, but with little margin. If S1\n      OOMs, set GRAD_CKPT=True (mathematically identical, ~30% slower)\n      or drop BATCH to 1 and raise GRAD_ACC to 4 to keep the exposure.")

In [ ]:
# %% 2 · forensic core — identical to the base study
def _c2(x,k):
    xt = torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt = torch.from_numpy(k)[None,None]
    if DEV=="cuda": return tF.conv2d(xt.cuda(), kt.cuda(), padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0 = np.float32(4.0)
    co = pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch = ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v = np.maximum(_c2(ch*ch, np.ones((w,w),np.float32)/(w*w)) - s0, 0.0)
                vm = v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n = pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r = n - n.mean(); sd = float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def load_lum(fp):
    with Image.open(fp) as im:
        a = np.asarray(im.convert("RGB"), np.float32)
    return (0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def ucl(x, alpha=None):
    a = C.ALPHA if alpha is None else alpha; k=len(x)
    return float(np.mean(x) + float(sps.t.ppf(1-a/2,df=k-1))*np.std(x,ddof=1)/np.sqrt(k))
def exact_signflip(m):
    obs=float(np.mean(m))
    return sum(1 for sg in itertools.product([-1,1],repeat=len(m))
               if float(np.mean(m*np.array(sg)))>=obs)/2**len(m)
print("[core] ready")

In [ ]:
# %% 3 · S0 — GATE: does our seed bank reproduce a stored generation bit for bit?
#
# The new adapters must generate from the SAME (prompt, seed) pairs as the existing six, or
# the paired design breaks and the twelve cluster means are not comparable. CAPTION and
# GEN_SEED_BASE above are the two ways to get that wrong, and neither fails loudly.
# This regenerates image 00000 from an ALREADY TRAINED adapter and compares bytes.
def s0():
    from diffusers import StableDiffusion3Pipeline
    if DEV != "cuda": HALT("S0 needs a GPU")
    ref = None
    for root in (BASE, PREV):
        for a in C.ARMS:
            for s in range(6):
                p = os.path.join(root,"gens",tag(a,s),"00000.png")
                if os.path.exists(p) and os.path.exists(
                        os.path.join(root,"adapters",tag(a,s),"pytorch_lora_weights.safetensors")):
                    ref = (root, tag(a,s), p); break
            if ref: break
        if ref: break
    if ref is None: HALT("found no existing adapter with a stored generation to check against")
    root, t, png = ref
    print(f"[S0] checking against {t} in {os.path.basename(root)}")
    pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True)
    pipe.load_lora_weights(os.path.join(root,"adapters",t))
    try:
        g = torch.Generator(device=DEV).manual_seed(C.GEN_SEED_BASE + 0)
        with torch.inference_mode():
            im = pipe(prompt=C.CAPTION, num_inference_steps=C.GEN_STEPS,
                      guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS,
                      generator=g).images[0]
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    new = np.asarray(im.convert("RGB"), np.int16)
    old = np.asarray(Image.open(png).convert("RGB"), np.int16)
    mad = float(np.abs(new-old).mean()); ident = bool((new==old).all())
    ajson({"reference":t,"root":root,"identical":ident,"mean_abs_diff":mad,
           "caption":C.CAPTION,"gen_seed_base":C.GEN_SEED_BASE},
          os.path.join(ROOT,"meta","S0_seedbank.json"), indent=2)
    print(f"[S0] bit-identical: {ident} | mean |diff|: {mad:.4f}/255")
    if ident:
        print("\n[S0] PASS — CAPTION and GEN_SEED_BASE match the base study. Proceed to S1.")
    elif mad < 1.0:
        print(f"\n[S0] NEAR MATCH ({mad:.3f}/255). Non-determinism across driver or library")
        print("     versions can do this. The seed bank is almost certainly right; record the")
        print("     value and proceed, but note it in the methods.")
    else:
        HALT(f"seed bank MISMATCH (mean |diff| {mad:.1f}/255).\n"
             f"CAPTION or GEN_SEED_BASE differs from the base study. Find the correct values "
             f"in\n{BASE}/meta/ or the pilot notebook's config before training anything — new "
             f"adapters\ngenerated from a different bank are NOT comparable to the existing six.")
if stage("S0"): s0()

In [ ]:
# %% 4 · S1 — train the twelve new adapters (seeds 6-11, both arms)
def _expected(t):
    _, a, s = next(x for x in NEW if x[0]==t)
    return {"tag":t,"arm":a,"seed":s,"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "model":C.SD,"dtype":str(DT)}
def avalid(t):
    if not os.path.exists(os.path.join(adir(t),"pytorch_lora_weights.safetensors")): return False
    p = os.path.join(adir(t),"train_meta.json")
    if not os.path.exists(p): return False
    try: m = json.load(open(p))
    except Exception: return False
    return all(m.get(k)==v for k,v in _expected(t).items()) and m.get("effective",False)

def s1():
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    for a in C.ARMS:
        n = len(glob.glob(os.path.join(tdir(a),"*.png")))
        if n == 0: HALT(f"training crops missing at {tdir(a)} — the base study must be present")
        print(f"[S1] arm {a}: {n} training crops from the base study")
    pp = os.path.join(ROOT,"adapters","_prompt.pt")
    if os.path.exists(pp):
        z = torch.load(pp, map_location=DEV); pe, po = z["pe"].to(DEV), z["pp"].to(DEV)
    else:
        pi = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
        with torch.no_grad():
            pe,_,po,_ = pi.encode_prompt(prompt=C.CAPTION, prompt_2=C.CAPTION,
                                         prompt_3=C.CAPTION, device=DEV,
                                         num_images_per_prompt=1,
                                         do_classifier_free_guidance=False)
        torch.save({"pe":pe.cpu(),"pp":po.cpu()}, pp); del pi
        gc.collect(); torch.cuda.empty_cache()

    for i,(t,a,seed) in enumerate(NEW,1):
        if avalid(t): print(f"[S1] {i}/{len(NEW)} skip valid {t}"); continue
        out = adir(t)
        if os.path.exists(out): shutil.move(out, out+f"__stale_{int(time.time())}")
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        try:
            pipe = StableDiffusion3Pipeline.from_pretrained(
                C.SD, torch_dtype=DT, token=True, text_encoder=None, text_encoder_2=None,
                text_encoder_3=None, tokenizer=None, tokenizer_2=None,
                tokenizer_3=None).to(DEV)
        except Exception:
            pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT,
                                                            token=True).to(DEV)
            pipe.text_encoder = pipe.text_encoder_2 = pipe.text_encoder_3 = None
        sch, vae, tr = pipe.scheduler, pipe.vae, pipe.transformer
        vae.requires_grad_(False); tr.requires_grad_(False)
        tr.add_adapter(LoraConfig(r=C.RANK, lora_alpha=C.RANK, init_lora_weights="gaussian",
                                  target_modules=list(C.TARGETS)))
        if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
        tr.train()
        torch.cuda.reset_peak_memory_stats()
        ps = [p for p in tr.parameters() if p.requires_grad]
        if not ps: HALT(f"{t}: LoRA did not attach")
        opt = torch.optim.AdamW(ps, lr=C.LR, weight_decay=1e-4)
        lrs = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=math.ceil(C.STEPS/C.GRAD_ACC))
        opt.zero_grad(set_to_none=True)
        files = sorted(glob.glob(os.path.join(tdir(a),"*.png")))
        cache = torch.stack([torch.from_numpy(
            np.asarray(Image.open(f).convert("RGB"), np.float32).copy()/127.5-1.0).permute(2,0,1)
            for f in files])
        vsf = float(vae.config.scaling_factor); vsh = float(vae.config.shift_factor or 0.0)
        ts, sg = sch.timesteps.to(DEV), sch.sigmas.to(DEV)
        nts = int(sch.config.num_train_timesteps)
        if len(sg)!=nts or len(ts)!=nts: HALT("scheduler array length mismatch — pin diffusers")
        g = torch.Generator().manual_seed(seed); losses=[]; t0=time.time()
        print(f"\n[S1] {i}/{len(NEW)} {t}: arm {a}, seed {seed}")
        for st in range(C.STEPS):
            idx = torch.randint(0, cache.shape[0], (C.BATCH,), generator=g)
            px = cache[idx].to(DEV, dtype=DT)
            with torch.no_grad():
                lat = vae.encode(px).latent_dist.sample(); lat = ((lat-vsh)*vsf).to(DT)
            noise = torch.randn_like(lat)
            u = torch.sigmoid(torch.randn(lat.shape[0], device=DEV, dtype=torch.float32))
            ii = (u*nts).long().clamp_(0,nts-1); sig = sg[ii].to(lat.dtype).view(-1,1,1,1)
            pred = tr(hidden_states=((1.0-sig)*lat + sig*noise).to(DT), timestep=ts[ii],
                      encoder_hidden_states=pe.repeat(lat.shape[0],1,1).to(DT),
                      pooled_projections=po.repeat(lat.shape[0],1).to(DT),
                      return_dict=False)[0]
            loss = tF.mse_loss(pred.float(), (noise-lat).float())/C.GRAD_ACC
            loss.backward()
            if (st+1)%C.GRAD_ACC==0:
                torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
                opt.zero_grad(set_to_none=True)
            losses.append(float(loss.detach())*C.GRAD_ACC)
            if not np.isfinite(losses[-1]): HALT(f"{t}: non-finite loss at step {st+1}")
            if st == 20:
                pk = torch.cuda.max_memory_allocated()/2**30
                tot = torch.cuda.get_device_properties(0).total_memory/2**30
                print(f"    peak after 20 steps: {pk:.1f} / {tot:.0f} GB "
                      f"({100*pk/tot:.0f}%)"
                      + ("   <- tight, consider GRAD_CKPT=True" if pk/tot > 0.85 else ""))
            if (st+1)%500==0:
                el=time.time()-t0
                print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-500:]):.5f} "
                      f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
        sd = get_peft_model_state_dict(tr)
        bn = float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
        if not (bn>1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
            HALT(f"{t}: DEAD adapter (||lora_B||={bn:.3e})")
        os.makedirs(out, exist_ok=True)
        StableDiffusion3Pipeline.save_lora_weights(save_directory=out, transformer_lora_layers=sd)
        m = _expected(t); m.update({"loss_tail":float(np.mean(losses[-100:])),
                                    "lora_B_norm":bn,"effective":True,
                                    "grad_ckpt":C.GRAD_CKPT,
                                    "peak_gb":float(torch.cuda.max_memory_allocated()/2**30),
                                    "minutes":(time.time()-t0)/60})
        ajson(m, os.path.join(out,"train_meta.json"), indent=2)
        print(f"[S1] saved {t}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
              f"({m['minutes']:.1f} min)")
        del pipe, tr, vae, cache; gc.collect(); torch.cuda.empty_cache()
    bad = [t for t,_,_ in NEW if not avalid(t)]
    if bad: HALT(f"S1 incomplete: {bad}")
    print(f"\n[S1] PASS — {len(NEW)} adapters")
    for t,_,_ in NEW:
        m = json.load(open(os.path.join(adir(t),"train_meta.json")))
        print(f"   {t:20s} ||lora_B|| {m['lora_B_norm']:7.2f}  tail {m['loss_tail']:.5f}")
if stage("S1"): s1()

In [ ]:
# %% 5 · S2 — generate 500 per adapter from the SAME seed bank
def s2():
    from diffusers import StableDiffusion3Pipeline
    if DEV != "cuda": HALT("S2 needs a GPU")
    pipe = StableDiffusion3Pipeline.from_pretrained(C.SD, torch_dtype=DT, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True); prev = False
    try:
        for t,_,_ in NEW:
            d = gdir(t); os.makedirs(d, exist_ok=True)
            for x in glob.glob(os.path.join(d,"*.tmp")): os.remove(x)
            miss = [i for i in range(C.G_PER)
                    if not os.path.exists(os.path.join(d,f"{i:05d}.png"))]
            if not miss: print(f"[S2] skip {t}"); continue
            if prev: pipe.unload_lora_weights(); prev=False
            pipe.load_lora_weights(adir(t)); prev=True
            print(f"[S2] {t}: {len(miss)} missing"); t0=time.time()
            torch.cuda.reset_peak_memory_stats()
            for s in range(0, len(miss), C.GEN_BATCH):
                ix = miss[s:s+C.GEN_BATCH]
                gs = [torch.Generator(device=DEV).manual_seed(C.GEN_SEED_BASE+i) for i in ix]
                with torch.inference_mode():
                    imgs = pipe(prompt=[C.CAPTION]*len(ix), num_inference_steps=C.GEN_STEPS,
                                guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS,
                                generator=gs).images
                for i,im in zip(ix,imgs):
                    tmp = os.path.join(d,f"{i:05d}.png.tmp")
                    im.save(tmp, format="PNG"); os.replace(tmp, os.path.join(d,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%40==0:
                    el=time.time()-t0
                    print(f"    {s+len(ix)}/{len(miss)}  eta "
                          f"{(el/max(s+len(ix),1))*(len(miss)-s-len(ix))/60:.0f}min")
            print(f"[S2] complete {t} ({(time.time()-t0)/60:.0f} min, peak "
                  f"{torch.cuda.max_memory_allocated()/2**30:.1f} GB)")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[S2] PASS")
if stage("S2"): s2()

In [ ]:
# %% 6 · S3 — measure. Column name matches the base study: rho_mult.
def s3():
    p = os.path.join(ROOT,"csv","sx2_measure.csv")
    done = set()
    if os.path.exists(p):
        for r in _csv.DictReader(open(p)): done.add((r["tag"], r["gen_idx"], r["K"]))
    for t,_,_ in NEW:
        todo = [i for i in range(C.G_PER)
                if not all((t,str(i),k) in done for k in C.ARMS)]
        if not todo: print(f"[S3] {t}: complete"); continue
        t0=time.time()
        for i in todo:
            Y = load_lum(os.path.join(gdir(t), f"{i:05d}.png")); W = wres(Y)
            for k in C.ARMS:
                if (t,str(i),k) in done: continue
                append_row(p, ["tag","gen_idx","K","rho_mult"],
                           [t, i, k, f"{ncc(W, Y*FP[k]):.6e}"])
        print(f"[S3] {t}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    d = pd.read_csv(p); exp = len(NEW)*C.G_PER*len(C.ARMS)
    if len(d) != exp: HALT(f"S3 incomplete: {len(d)}/{exp}")
    print(f"[S3] PASS — {len(d)} rows")
if stage("S3"): s3()

In [ ]:
# %% 7 · S4 — pooled analysis at k = 12. DO NOT RUN UNTIL S1-S3 ARE COMPLETE.
def _valcol(df):
    for c_ in ("rho_mult","rho","corr","ncc"):
        if c_ in df.columns: return c_
    num=[c_ for c_ in df.columns if df[c_].dtype.kind=="f" and df[c_].abs().max()<=1.5]
    return num[-1] if num else None

def s4():
    frames = []
    for root, name in ((BASE,"s5_measure.csv"), (PREV,"b3_measure.csv"),
                       (ROOT,"sx2_measure.csv")):
        f = os.path.join(root,"csv",name)
        if os.path.exists(f):
            df = pd.read_csv(f)
            for c_ in df.columns:
                if c_ in ("tag","K"): df[c_] = df[c_].astype(str)
            frames.append((os.path.basename(root), df, _valcol(df)))
            print(f"[S4] {os.path.basename(root)}/{name}: {len(df)} rows, "
                  f"value column '{_valcol(df)}'")
    if len(frames) < 3: HALT("need all three measurement roots to pool to k=12")

    per = {a: {} for a in C.ARMS}
    for _, df, vc in frames:
        for a in C.ARMS:
            oth = [o for o in C.ARMS if o != a][0]
            for t in df.tag.unique():
                ts = str(t)
                if not (ts.startswith(f"{a}_raw_s") and ts.endswith("_r16")): continue
                g = df[df.tag==t]
                own = g[g.K==a].sort_values("gen_idx")[vc].to_numpy(float)
                ot  = g[g.K==oth].sort_values("gen_idx")[vc].to_numpy(float)
                n = min(len(own), len(ot))
                if n: per[a][ts] = float((own[:n]-ot[:n]).mean())
    kA, kB = len(per["A"]), len(per["B"])
    print(f"\n[S4] arm A: {kA} adapters {sorted(per['A'])}")
    print(f"     arm B: {kB} adapters {sorted(per['B'])}")
    if kA != kB: print(f"     *** arms differ in size ({kA} vs {kB}) — the limit uses each arm's own k")

    R_REAL = 3.56703416571125e-02
    out = {"k_A":kA, "k_B":kB, "R_real":R_REAL, "per_adapter":
           {a:{t:v for t,v in sorted(per[a].items())} for a in C.ARMS}}
    print(f"\n{'arm':>4}{'k':>4}{'mean':>13}{'sd':>12}{'U':>13}{'exact p':>11}{'floor':>10}")
    print("-"*67)
    for a in C.ARMS:
        v = np.array([per[a][t] for t in sorted(per[a])])
        U = ucl(v); pv = exact_signflip(v)
        out[a] = {"mean":float(v.mean()), "sd":float(v.std(ddof=1)), "U":U,
                  "exact_signflip_p":pv, "floor":1/2**len(v),
                  "shapiro_p":float(sps.shapiro(v).pvalue),
                  "kurtosis":float(sps.kurtosis(v))}
        print(f"{a:>4}{len(v):>4}{v.mean():>13.4e}{v.std(ddof=1):>12.3e}{U:>13.4e}"
              f"{pv:>11.4f}{1/2**len(v):>10.5f}")
    U = max(out[a]["U"] for a in C.ARMS)
    sym = np.mean([np.mean([per[a][t] for t in sorted(per[a])]) for a in C.ARMS])
    out.update({"U_device":U, "lambda_U_pct":100*U/R_REAL, "theta_sym":float(sym)})
    ajson(out, os.path.join(ROOT,"S4_bound.json"), indent=2)

    print(f"\n  U_device      {U:.4e}")
    print(f"  lambda_U      {100*U/R_REAL:.4f}%   (was 0.2490% at k=6)")
    print(f"  theta_sym     {sym:+.4e}   (was +1.494e-05 at k=6)")
    print(f"\n  cluster distribution, now testable with k>6:")
    for a in C.ARMS:
        print(f"    arm {a}: Shapiro p {out[a]['shapiro_p']:.3f}, "
              f"excess kurtosis {out[a]['kurtosis']:+.2f}")
    print("""
  Shapiro p below 0.05 or kurtosis well above 0 means the normal-model t limit is the
  wrong reference and the empirical or heavy-tailed row of the coverage simulation is the
  one to cite. This is the check six clusters could not support.""")

    k = min(kA, kB)
    print("\n" + "="*72)
    if 1/2**k < C.ALPHA:
        rej = all(out[a]["exact_signflip_p"] < C.ALPHA for a in C.ARMS)
        print(f"THE REGISTERED TEST IS NOW CAPABLE OF REJECTING (floor 1/{2**k} < "
              f"{C.ALPHA}).")
        print(f"It {'REJECTS' if rej else 'DOES NOT REJECT'}: "
              f"p_A = {out['A']['exact_signflip_p']:.4f}, "
              f"p_B = {out['B']['exact_signflip_p']:.4f}.")
        if not rej:
            print("\nThat is the sentence the study could not previously write: the registered")
            print("intersection-union test could reject at the prespecified level and did not.")
            print("Report it as the primary result; the upper limit becomes supporting evidence")
            print("rather than a post-registration substitute.")
    else:
        print(f"floor 1/{2**k} still exceeds {C.ALPHA} — more adapters needed.")
    print("="*72)
if stage("S4"): s4()

## Running it

Set `C.STAGES` and re-run cell 1 each time.

| stage | wall | what it does |
|---|---|---|
| `("S0",)` | ~2 min | **gate** — regenerates a stored image and compares bytes |
| `("S1",)` | ~5 h | trains the twelve new adapters |
| `("S2",)` | ~6 h | 500 generations each, same seed bank |
| `("S3",)` | ~25 min | measurement |
| `("S4",)` | seconds | pooled analysis at k = 12 |

### S0 is not optional
`CAPTION` and `GEN_SEED_BASE` in cell 1 are the two settings that can silently break the
paired design, and neither fails loudly — you would get twelve perfectly good adapters
whose generations are not comparable to the existing six. S0 regenerates image `00000` from
an already-trained adapter and compares it to the stored PNG. Bit-identical means both
settings are right.

If S0 halts, find the correct values in `E_INV_P0_v3/meta/` or the pilot notebook's config
before training anything.

### Resuming
Everything is keyed on what exists. Adapters skip when `train_meta.json` validates,
generations skip per file, measurements skip per `(tag, gen_idx, K)` row. A dropped session
loses at most one adapter's progress.

### After S4
Rerun `verify_einv.py` and `05_track_a_bounds.ipynb` for the bootstrap at the new k, then
update the manuscript. The numbers that change: `t_{0.995,k-1}` 4.032 → 3.106, `U_device`,
both normalized limits, the sign-flip floor, and Table II's coverage discussion — BCa is no
longer marginal at twelve clusters.